# Interactive notebook: planetary accelerations acting on an asteroid

This notebook creates an **expandable, widget-based plot** of acceleration amplitudes acting on an asteroid in the Solar System.

## Included in this updated version
For an asteroid orbiting the Sun, the notebook now includes:

- **Sun's direct central acceleration**
- **planets total acceleration**
- **planets perturbing gravitational acceleration in heliocentric frame**


## Orbit assumptions in this version
- Asteroid orbit: **coplanar Keplerian ellipse** with interactive sliders for:
  - semimajor axis `a_ast`
  - eccentricity `e_ast`
- Planets: **coplanar circular heliocentric orbits**
- x-axis: **time over one synodic cycle with Jupiter** in **days**
- y-axis: **acceleration amplitude** in **m/s²**, on a **logarithmic scale**

Authors: Gijs Verdoes Kleijn (g.a.verdoes.kleijn@rug.nl) assisted by [ChatGPT](https://chatgpt.com/share/69d22e0f-146c-8396-ab96-69481fa79786)


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Physical constants
G = 6.67430e-11                 # m^3 kg^-1 s^-2
AU = 1.495978707e11             # m
DAY = 86400.0                  # s

M_SUN = 1.98847e30             # kg
MU_SUN = G * M_SUN

BODY_DATA = {
    "Venus":   {"mass": 4.8675e24,  "a_au": 0.723332,  "phase_deg": 0.0},
    "Earth":    {"mass": 6.0e24,  "a_au": 1.0,  "phase_deg": 0.0},
    "Mars":    {"mass": 6.4171e23,  "a_au": 1.523679,  "phase_deg": 0.0},
    "Jupiter": {"mass": 1.89813e27, "a_au": 5.2044,    "phase_deg": 0.0},
    "Saturn":  {"mass": 5.6834e26,  "a_au": 9.5826,    "phase_deg": 0.0},
}
for body in BODY_DATA:
    BODY_DATA[body]["mu"] = G * BODY_DATA[body]["mass"]

DEFAULTS = {
    "a_ast_au": 2.0,
    "e_ast": 0.0,
    "n_samples": 1800,
}

def circular_mean_motion(a_m, mu=MU_SUN):
    return np.sqrt(mu / a_m**3)

def synodic_period_days(a_ast_au=1.0, a_jup_au=5.2044):
    n_ast = circular_mean_motion(a_ast_au * AU)
    n_jup = circular_mean_motion(a_jup_au * AU)
    return 2.0 * np.pi / abs(n_ast - n_jup) / DAY

def solve_kepler(M, e, tol=1e-12, maxiter=50):
    M = np.asarray(M)
    E = np.array(M, dtype=float)
    if e > 0.8:
        E[:] = np.pi
    for _ in range(maxiter):
        f = E - e * np.sin(E) - M
        fp = 1.0 - e * np.cos(E)
        dE = -f / fp
        E = E + dE
        if np.max(np.abs(dE)) < tol:
            break
    return E

def asteroid_position_elliptic(a_au, e, t_sec, M0_deg=0.0):
    a_m = a_au * AU
    n = circular_mean_motion(a_m)
    M = n * t_sec + np.deg2rad(M0_deg)
    E = solve_kepler(M, e)
    x = a_m * (np.cos(E) - e)
    y = a_m * np.sqrt(1.0 - e**2) * np.sin(E)
    return np.stack([x, y], axis=0)

def circular_position(a_au, theta_rad):
    r = a_au * AU
    return np.stack([r * np.cos(theta_rad), r * np.sin(theta_rad)], axis=0)

def body_position_circular(a_au, t_sec, phase_deg=0.0):
    n = circular_mean_motion(a_au * AU)
    theta = n * t_sec + np.deg2rad(phase_deg)
    return circular_position(a_au, theta)

def acceleration_sun_on_asteroid(r_ast_vec):
    r = np.linalg.norm(r_ast_vec, axis=0)
    return MU_SUN / r**2

def total_accel_from_body_on_asteroid(body_mu, r_ast_vec, r_body_vec):
    delta = r_body_vec - r_ast_vec
    d = np.linalg.norm(delta, axis=0)
    return body_mu / d**2

def perturbing_accel_in_heliocentric_frame(body_mu, r_ast_vec, r_body_vec):
    delta = r_body_vec - r_ast_vec
    d = np.linalg.norm(delta, axis=0)
    rb = np.linalg.norm(r_body_vec, axis=0)
    term1 = delta / d**3
    term2 = r_body_vec / rb**3
    a_vec = body_mu * (term1 - term2)
    return np.linalg.norm(a_vec, axis=0)

def compute_time_series(a_ast_au=DEFAULTS['a_ast_au'], e_ast=0.0, n_samples=1800):
    print(a_ast_au)
    T_syn_days = synodic_period_days(a_ast_au=a_ast_au, a_jup_au=BODY_DATA["Jupiter"]["a_au"])
    t_sec = np.linspace(0.0, T_syn_days * DAY, n_samples)

    r_ast = asteroid_position_elliptic(a_ast_au, e_ast, t_sec, M0_deg=0.0)
    a_sun = acceleration_sun_on_asteroid(r_ast)

    result = {
        "t_days": t_sec / DAY,
        "T_syn_days": T_syn_days,
        "r_ast": r_ast,
        "a_sun": a_sun,
    }

    for name, data in BODY_DATA.items():
        r_body = body_position_circular(data["a_au"], t_sec, phase_deg=data["phase_deg"])
        result[f"{name}_r"] = r_body
        result[f"{name}_total"] = total_accel_from_body_on_asteroid(data["mu"], r_ast, r_body)
        result[f"{name}_pert"] = perturbing_accel_in_heliocentric_frame(data["mu"], r_ast, r_body)

    return result

def summary_numbers(a_ast_au=2.0, e_ast=0.0):
    d = compute_time_series(a_ast_au=a_ast_au, e_ast=e_ast, n_samples=1200)
    out = {
        "synodic_days": d["T_syn_days"],
        "sun_min": float(np.min(d["a_sun"])),
        "sun_max": float(np.max(d["a_sun"])),
    }
    for name in BODY_DATA:
        out[f"{name}_total_min"] = float(np.min(d[f"{name}_total"]))
        out[f"{name}_total_max"] = float(np.max(d[f"{name}_total"]))
        out[f"{name}_pert_min"] = float(np.min(d[f"{name}_pert"]))
        out[f"{name}_pert_max"] = float(np.max(d[f"{name}_pert"]))
    return out


In [5]:
import ipywidgets
ipywidgets.__version__

'8.1.8'

In [7]:
import ipywidgets as widgets
widgets.IntSlider()

IntSlider(value=0)

## Quick numerical check
This reports the synodic period with Jupiter and the rough acceleration ranges for the current asteroid orbit.


In [10]:
summary_numbers()


2.0


{'synodic_days': 1356.159141972799,
 'sun_min': 0.0014825657108111301,
 'sun_max': 0.0014825657108111315,
 'Venus_total_min': 1.9573111853876453e-09,
 'Venus_total_max': 8.906455741414071e-09,
 'Venus_pert_min': 2.5787739543615262e-08,
 'Venus_pert_max': 3.665150515269679e-08,
 'Earth_total_min': 1.9882200932178093e-09,
 'Earth_total_max': 1.789394713496665e-08,
 'Earth_pert_min': 1.5905731254751456e-08,
 'Earth_pert_max': 3.578789426993331e-08,
 'Mars_total_min': 1.541349839244472e-10,
 'Mars_total_max': 8.43517794145564e-09,
 'Mars_pert_min': 6.659810970249571e-10,
 'Mars_pert_max': 9.259518641576859e-09,
 'Jupiter_total_min': 1.0906510345673999e-07,
 'Jupiter_total_max': 5.512992564129087e-07,
 'Jupiter_pert_min': 7.607028011747908e-08,
 'Jupiter_pert_max': 3.423025142241923e-07,
 'Saturn_total_min': 1.2634297102262575e-08,
 'Saturn_total_max': 2.9479955895051393e-08,
 'Saturn_pert_min': 3.790429073409624e-09,
 'Saturn_pert_max': 1.1021456044256793e-08}

## Interactive plot

Use the sliders below to change the asteroid orbit.

### Asteroid orbit controls
- `a_ast [au]`: semimajor axis of the asteroid orbit
- `e_ast`: eccentricity of the asteroid orbit

### Curves
For each planet, you can toggle:
- **total** gravitational acceleration on the asteroid
- **perturbing** heliocentric-frame acceleration

The Sun direct acceleration is always the dominant central term and is also toggleable.


In [21]:
a_ast_widget = widgets.FloatSlider(
    value=DEFAULTS["a_ast_au"], min=0.4, max=5.0, step=0.01, description="a_ast [au]",
    continuous_update=False, readout_format=".2f",
    style={"description_width": "initial"}, layout=widgets.Layout(width="330px")
)

e_ast_widget = widgets.FloatSlider(
    value=DEFAULTS["e_ast"], min=0.0, max=0.95, step=0.01, description="e_ast",
    continuous_update=False, readout_format=".2f",
    style={"description_width": "initial"}, layout=widgets.Layout(width="330px")
)

samples_widget = widgets.IntSlider(
    value=DEFAULTS["n_samples"], min=300, max=5000, step=100, description="n_samples",
    continuous_update=False, style={"description_width": "initial"}, layout=widgets.Layout(width="330px")
)

show_sun = widgets.Checkbox(value=True, description="Sun direct", indent=False)

show_venus_total = widgets.Checkbox(value=True, description="Venus total", indent=False)
#show_venus_pert = widgets.Checkbox(value=True, description="Venus perturbing", indent=False)

show_earth_total = widgets.Checkbox(value=True, description="Earth total", indent=False)
#show_earth_pert = widgets.Checkbox(value=True, description="Earth perturbing", indent=False)

show_mars_total = widgets.Checkbox(value=True, description="Mars total", indent=False)
#show_mars_pert = widgets.Checkbox(value=True, description="Mars perturbing", indent=False)

show_jupiter_total = widgets.Checkbox(value=True, description="Jupiter total", indent=False)
#show_jupiter_pert = widgets.Checkbox(value=True, description="Jupiter perturbing", indent=False)

show_saturn_total = widgets.Checkbox(value=True, description="Saturn total", indent=False)
#show_saturn_pert = widgets.Checkbox(value=True, description="Saturn perturbing", indent=False)

out = widgets.Output()

def draw_plot(*args):
    with out:
        clear_output(wait=True)

        a_ast = a_ast_widget.value
        e_ast = e_ast_widget.value
        n_samples = samples_widget.value

        d = compute_time_series(a_ast_au=a_ast, e_ast=e_ast, n_samples=n_samples)

        fig, ax = plt.subplots(figsize=(11, 7))

        if show_sun.value:
            ax.plot(d["t_days"], d["a_sun"], linewidth=2, label="Sun direct")

        if show_venus_total.value:
            ax.plot(d["t_days"], d["Venus_total"], linewidth=1.8, label="Venus total")
#        if show_venus_pert.value:
#            ax.plot(d["t_days"], d["Venus_pert"], linewidth=1.8, label="Venus perturbing")

        if show_earth_total.value:
            ax.plot(d["t_days"], d["Earth_total"], linewidth=1.8, label="Earth total")
#        if show_earth_pert.value:
#            ax.plot(d["t_days"], d["Earth_pert"], linewidth=1.8, label="Earth perturbing")

        if show_mars_total.value:
            ax.plot(d["t_days"], d["Mars_total"], linewidth=1.8, label="Mars total")
#        if show_mars_pert.value:
#            ax.plot(d["t_days"], d["Mars_pert"], linewidth=1.8, label="Mars perturbing")

        if show_jupiter_total.value:
            ax.plot(d["t_days"], d["Jupiter_total"], linewidth=1.8, label="Jupiter total")
#        if show_jupiter_pert.value:
#            ax.plot(d["t_days"], d["Jupiter_pert"], linewidth=1.8, label="Jupiter perturbing")

        if show_saturn_total.value:
            ax.plot(d["t_days"], d["Saturn_total"], linewidth=1.8, label="Saturn total")
#        if show_saturn_pert.value:
#            ax.plot(d["t_days"], d["Saturn_pert"], linewidth=1.8, label="Saturn perturbing")

        ax.set_yscale("log")
        ax.set_xlabel("Time over one Jupiter synodic cycle [days]")
        ax.set_ylabel("Acceleration amplitude [m/s²]")
        ax.set_title("Accelerations acting on an asteroid: Sun, Venus, Mars, Jupiter, Saturn")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(ncol=2)

        syn = d["T_syn_days"]
        text = (
            f"a_ast={a_ast:.3f} au, e_ast={e_ast:.3f}, "
            f"Jupiter synodic period={syn:.2f} d"
        )
        ax.text(
            0.02, 0.02, text, transform=ax.transAxes,
            fontsize=9, va="bottom", ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
        )
        plt.legend(loc='upper right')
        plt.show()

        print(f"Jupiter synodic period: {syn:.6f} days")
        print(f"Sun direct acceleration range: {np.min(d['a_sun']):.6e} .. {np.max(d['a_sun']):.6e} m/s²")
        for name in ["Venus", "Mars", "Jupiter", "Saturn"]:
            print(f"{name} total range: {np.min(d[f'{name}_total']):.6e} .. {np.max(d[f'{name}_total']):.6e} m/s²")
            print(f"{name} perturbing range: {np.min(d[f'{name}_pert']):.6e} .. {np.max(d[f'{name}_pert']):.6e} m/s²")

for w in [
    a_ast_widget, e_ast_widget, samples_widget, show_sun,
    show_venus_total, show_venus_pert, show_earth_total, show_earth_pert,
    show_mars_total, show_mars_pert,
    show_jupiter_total, show_jupiter_pert,
    show_saturn_total, show_saturn_pert
]:
    w.observe(draw_plot, names="value")

controls = widgets.VBox([
    widgets.HBox([a_ast_widget, e_ast_widget, samples_widget]),
    widgets.HBox([show_sun]),
    widgets.HBox([show_venus_total, show_venus_pert, show_mars_total, show_mars_pert]),
    widgets.HBox([show_earth_total, show_earth_pert]),
    widgets.HBox([show_jupiter_total, show_jupiter_pert, show_saturn_total, show_saturn_pert]),
])

display(controls, out)
draw_plot()


Output()

## Notes for the next extension

Natural next steps:
* Add **solar $J_2$**
* Add **GR terms**
* Add a second panel with **distance to the perturber** or **radial/transverse perturbation**
* Replace circular-planet orbits with **full Keplerian elliptical orbits**
* Build a final multi-curve perturbation-order plot more analogous to this [book figure](https://drive.google.com/file/d/1_G9eufQTa3xELmdVL6EFRTxrM3B7NmlV/edit)
